# 02 — Data Cleaning & Enrichment
**Goal:** Join provider info and HCPCS descriptions, filter bad data, and create derived columns.

**Steps:**
1. Narrow `provider_info` to 25 useful columns
2. Join HCPCS descriptions
3. Filter malformed HCPCS codes
4. Create derived metrics
5. Export clean Parquet files for analysis

In [ ]:
import duckdb
import polars as pl
import os

con = duckdb.connect()

# ── Configure paths ────────────────────────────────────────────────────────────
# Update these to match your local setup, or use the Streamlit Settings page
# to save them to .medicaid_config.json and load them here automatically.

import sys, json
from pathlib import Path

_config_path = Path(__file__).parent.parent / ".medicaid_config.json"
_cfg = {}
if _config_path.exists():
    _cfg = json.loads(_config_path.read_text())

parquet_path  = _cfg.get("parquet_path",  os.path.expanduser("~/Downloads/medicaid-provider-spending.parquet"))
csv_path      = _cfg.get("nppes_csv_path", os.path.expanduser("~/Downloads/npidata.csv"))
hcpcs_csv_path= _cfg.get("hcpcs_csv_path", "")
exports_dir   = _cfg.get("exports_dir",    str(Path(__file__).parent.parent / "exports"))

print("Medicaid parquet :", parquet_path)
print("NPPES CSV        :", csv_path)
print("HCPCS CSV        :", hcpcs_csv_path or "(not set — descriptions will be NULL)")
print("Exports dir      :", exports_dir)

con.execute(f"CREATE VIEW medicaid AS SELECT * FROM read_parquet('{parquet_path}')")

## 1. Narrow Provider Info to Useful Columns
Skipping SUMMARIZE — columns already identified (< 50% null threshold).

In [2]:
final_columns = [
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Credential Text",
    "Provider First Line Business Mailing Address",
    "Provider Business Mailing Address City Name",
    "Provider Business Mailing Address State Name",
    "Provider Business Mailing Address Postal Code",
    "Provider Business Mailing Address Country Code (If outside U.S.)",
    "Provider Business Mailing Address Telephone Number",
    "Provider First Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)",
    "Provider Business Practice Location Address Telephone Number",
    "Provider Enumeration Date",
    "Last Update Date",
    "Provider Sex Code",
    "Healthcare Provider Taxonomy Code_1",
    "Provider License Number_1",
    "Provider License Number State Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
    "Is Sole Proprietor",
]

cols_sql = ", ".join([f'"{c}"' for c in final_columns])

con.execute(
    f"CREATE OR REPLACE VIEW provider_info AS SELECT {cols_sql} FROM read_csv_auto('{csv_path}', all_varchar=1)"
)
print("provider_info view created with", len(final_columns), "columns")

provider_info view created with 25 columns


## 2. Load & Filter HCPCS Codes
Keep only clean 5-character codes (Level I CPT + Level II).

In [ ]:
# ── HCPCS codes (optional enrichment) ─────────────────────────────────────────
# Download from CMS:
#   https://www.cms.gov/medicare/coding-billing/healthcare-common-procedure-system
# Expected columns: HCPC, SHORT DESCRIPTION, LONG DESCRIPTION
#
# The Settings page (⚙️) in the Streamlit app will save this path automatically.

if hcpcs_csv_path and os.path.exists(hcpcs_csv_path):
    hcpcs_codes = pl.read_csv(hcpcs_csv_path, infer_schema_length=0)
    con.register("hcpcs_codes", hcpcs_codes)
    print(f"Loaded {len(hcpcs_codes):,} HCPCS codes from {hcpcs_csv_path}")
else:
    print("HCPCS CSV not set or not found — creating empty placeholder.")
    print("Descriptions will be NULL. Set hcpcs_csv_path above to enable enrichment.")
    con.execute(
        'CREATE TABLE IF NOT EXISTS hcpcs_codes (HCPC VARCHAR, "SHORT DESCRIPTION" VARCHAR, "LONG DESCRIPTION" VARCHAR)'
    )

## 3. Create Enriched View
Zero RAM cost — DuckDB computes this on demand.

In [4]:
con.execute("""
    CREATE OR REPLACE VIEW medicaid_enriched AS
    SELECT
        m.BILLING_PROVIDER_NPI_NUM,
        m.HCPCS_CODE,
        h."LONG DESCRIPTION"             AS hcpcs_description,
        m.CLAIM_FROM_MONTH,
        SUBSTR(m.CLAIM_FROM_MONTH, 1, 4) AS year,
        SUBSTR(m.CLAIM_FROM_MONTH, 6, 2) AS month,
        m.TOTAL_UNIQUE_BENEFICIARIES,
        m.TOTAL_CLAIMS,
        m.TOTAL_PAID,
        ROUND(m.TOTAL_PAID / NULLIF(m.TOTAL_CLAIMS, 0), 2)              AS avg_paid_per_claim,
        ROUND(m.TOTAL_CLAIMS / NULLIF(m.TOTAL_UNIQUE_BENEFICIARIES, 0), 2) AS avg_claims_per_beneficiary,
        p."Provider Last Name (Legal Name)"                             AS last_name,
        p."Provider First Name"                                         AS first_name,
        p."Provider Business Practice Location Address State Name"      AS state,
        p."Healthcare Provider Taxonomy Code_1"                         AS taxonomy_code,
        p."Entity Type Code"                                            AS entity_type
    FROM medicaid m
    LEFT JOIN hcpcs_codes h ON m.HCPCS_CODE = h.HCPC
    LEFT JOIN provider_info p ON m.BILLING_PROVIDER_NPI_NUM = p.NPI
    WHERE LENGTH(m.HCPCS_CODE) = 5
""")
print("medicaid_enriched view created")

CatalogException: Catalog Error: Table with name hcpcs_codes does not exist!
Did you mean "pg_indexes"?

LINE 21:     LEFT JOIN hcpcs_codes h ON m.HCPCS_CODE = h.HCPC
                       ^

## 4. Export Parquet Files for Analysis & Tableau

In [ ]:
# ── Create exports dir ─────────────────────────────────────────────────────────
os.makedirs(exports_dir, exist_ok=True)
print("Exports directory:", exports_dir)

# ── by_state_month.parquet ─────────────────────────────────────────────────────
by_state_path = os.path.join(exports_dir, "by_state_month.parquet")
con.execute(f"""
    COPY (
        SELECT
            state, year, month,
            SUM(TOTAL_PAID)                          AS total_paid,
            SUM(TOTAL_CLAIMS)                        AS total_claims,
            SUM(TOTAL_UNIQUE_BENEFICIARIES)          AS total_beneficiaries,
            COUNT(DISTINCT BILLING_PROVIDER_NPI_NUM) AS unique_providers
        FROM medicaid_enriched
        GROUP BY state, year, month
        ORDER BY year, month, state
    ) TO '{by_state_path}' (FORMAT 'PARQUET')
""")
print("Exported:", by_state_path)

In [ ]:
# ── by_hcpcs_month.parquet ─────────────────────────────────────────────────────
by_hcpcs_path = os.path.join(exports_dir, "by_hcpcs_month.parquet")
con.execute(f"""
    COPY (
        SELECT
            HCPCS_CODE, hcpcs_description, year, month,
            SUM(TOTAL_PAID)   AS total_paid,
            SUM(TOTAL_CLAIMS) AS total_claims,
            ROUND(SUM(TOTAL_PAID) / NULLIF(SUM(TOTAL_CLAIMS), 0), 2) AS avg_paid_per_claim
        FROM medicaid_enriched
        GROUP BY HCPCS_CODE, hcpcs_description, year, month
        ORDER BY total_paid DESC
    ) TO '{by_hcpcs_path}' (FORMAT 'PARQUET')
""")
print("Exported:", by_hcpcs_path)

In [ ]:
# ── top_providers.parquet ──────────────────────────────────────────────────────
top_providers_path = os.path.join(exports_dir, "top_providers.parquet")
con.execute(f"""
    COPY (
        SELECT
            BILLING_PROVIDER_NPI_NUM AS npi,
            last_name, first_name, state, taxonomy_code, entity_type,
            SUM(TOTAL_PAID)                          AS total_paid,
            SUM(TOTAL_CLAIMS)                        AS total_claims,
            SUM(TOTAL_UNIQUE_BENEFICIARIES)          AS total_beneficiaries,
            ROUND(SUM(TOTAL_PAID) / NULLIF(SUM(TOTAL_CLAIMS), 0), 2)              AS avg_paid_per_claim,
            ROUND(SUM(TOTAL_CLAIMS) / NULLIF(SUM(TOTAL_UNIQUE_BENEFICIARIES), 0), 2) AS avg_claims_per_beneficiary
        FROM medicaid_enriched
        GROUP BY npi, last_name, first_name, state, taxonomy_code, entity_type
        ORDER BY total_paid DESC
        LIMIT 10000
    ) TO '{top_providers_path}' (FORMAT 'PARQUET')
""")
print("Exported:", top_providers_path)
print("\\nAll exports complete. Files written to:", exports_dir)